# LangGraph Stream 输出：从状态更新到调试事件

> 适用版本：本项目锁定的 **LangGraph 1.1.2**、**LangChain 1.2.12**、**langchain-core 1.2.18** 与 **langchain-deepseek 1.0.1**。本教程只讨论本地已编译图的 Python `stream()` / `astream()`，不涉及 Agent Server、远程 SDK、SSE 或前端渲染。

普通 `invoke()` 只返回图完成后的结果；流式 API 则让调用方在图运行期间持续收到状态、模型消息、业务进度或运行时诊断事件。LangGraph 1.1.2 提供七种 `stream_mode`：

| 模式 | 主要输出 | 典型用途 |
| --- | --- | --- |
| `values` | 每个 step 后的完整 State | 展示当前整体状态 |
| `updates` | 节点本次返回的 State 增量 | 展示工作流进度 |
| `messages` | `AIMessageChunk` 与模型调用 metadata | 实时显示模型文本 |
| `custom` | 节点主动写出的任意业务数据 | 进度条、阶段说明 |
| `checkpoints` | checkpoint 创建事件 | 持久化时间线观察 |
| `tasks` | task 开始与完成事件 | 节点级执行观察 |
| `debug` | checkpoint、task 与额外调试信息 | 深度排障 |

本教程以 `version="v2"` 为主。v2 把不同模式统一为带 `type`、`ns`、`data` 的 `StreamPart` 字典；但 LangGraph 1.1.2 的默认值仍是 v1，因此后文会专门比较两者。

## 总体流程

```mermaid
flowchart LR
    I[输入 State] --> G[Compiled Graph]
    G --> N1[节点或任务运行]
    N1 --> E{启用的 stream_mode}
    E --> V[values / updates]
    E --> M[messages]
    E --> C[custom]
    E --> R[checkpoints / tasks / debug]
    V --> P[StreamPart v2]
    M --> P
    C --> P
    R --> P
    P --> U[调用方逐个消费事件]
```

流式输出不会改变图的业务结果。它只是把执行过程中的不同观察面投影为迭代器或异步迭代器。

In [1]:
import inspect
import os
from importlib.metadata import version
from pprint import pprint
from typing import TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessageChunk, HumanMessage
from langchain_core.runnables.schema import StreamEvent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, MessagesState, StateGraph


versions = {
    package: version(package)
    for package in (
        "langgraph",
        "langchain",
        "langchain-core",
        "langchain-deepseek",
    )
}
print("当前实际运行版本（完整原始字典）：")
pprint(versions)

assert versions == {
    "langgraph": "1.1.2",
    "langchain": "1.2.12",
    "langchain-core": "1.2.18",
    "langchain-deepseek": "1.0.1",
}


当前实际运行版本（完整原始字典）：
{'langchain': '1.2.12',
 'langchain-core': '1.2.18',
 'langchain-deepseek': '1.0.1',
 'langgraph': '1.1.2'}


## 1. 先建立一个可重复运行的基础图

基础图包含 `normalize_topic → build_result` 两个节点：

- 两个节点都返回局部 State 更新；
- 两个节点都用 `get_stream_writer()` 主动写出业务进度；
- 图本身不调用模型，也不依赖 checkpointer，适合反复比较输出协议。

同一张图会被多次调用。每次都使用相同输入，以便只观察 `stream_mode` 带来的差异。

In [2]:
class WorkflowState(TypedDict):
    topic: str
    normalized_topic: str
    result: str


def normalize_topic(state: WorkflowState) -> dict[str, str]:
    writer = get_stream_writer()
    writer(
        {
            "event": "progress",
            "node": "normalize_topic",
            "message": "正在规范化主题",
        }
    )
    return {"normalized_topic": state["topic"].strip().lower()}


def build_result(state: WorkflowState) -> dict[str, str]:
    writer = get_stream_writer()
    writer(
        {
            "event": "progress",
            "node": "build_result",
            "message": "正在生成最终结果",
        }
    )
    return {"result": f"已处理主题：{state['normalized_topic']}"}


workflow_builder = StateGraph(WorkflowState)
workflow_builder.add_node("normalize_topic", normalize_topic)
workflow_builder.add_node("build_result", build_result)
workflow_builder.add_edge(START, "normalize_topic")
workflow_builder.add_edge("normalize_topic", "build_result")
workflow_builder.add_edge("build_result", END)
workflow_graph = workflow_builder.compile()

workflow_input: WorkflowState = {
    "topic": "  LangGraph Stream  ",
    "normalized_topic": "",
    "result": "",
}

print("基础图输入（完整原始对象）：")
pprint(workflow_input)


基础图输入（完整原始对象）：
{'normalized_topic': '', 'result': '', 'topic': '  LangGraph Stream  '}


## 2. `values`：观察每一步后的完整 State

`values` 会先发出输入 State，然后在每个 step 合并节点更新后再次发出完整 State。它适合直接刷新界面上的整体数据，但 State 很大时也会重复传输大量未变化字段。

In [3]:
values_events = list(
    workflow_graph.stream(
        workflow_input,
        stream_mode="values",
        version="v2",
    )
)

print("values 模式的全部原始 StreamPart：")
for index, event in enumerate(values_events, start=1):
    print(f"\n--- values event {index} ---")
    pprint(event)


values 模式的全部原始 StreamPart：

--- values event 1 ---
{'data': {'normalized_topic': '',
          'result': '',
          'topic': '  LangGraph Stream  '},
 'interrupts': (),
 'ns': (),
 'type': 'values'}

--- values event 2 ---
{'data': {'normalized_topic': 'langgraph stream',
          'result': '',
          'topic': '  LangGraph Stream  '},
 'interrupts': (),
 'ns': (),
 'type': 'values'}

--- values event 3 ---
{'data': {'normalized_topic': 'langgraph stream',
          'result': '已处理主题：langgraph stream',
          'topic': '  LangGraph Stream  '},
 'interrupts': (),
 'ns': (),
 'type': 'values'}


In [4]:
# 机制验证与原始观察分开：只检查稳定的协议和 State 变化。
assert len(values_events) == 3
assert all(event["type"] == "values" for event in values_events)
assert all(event["ns"] == () for event in values_events)
assert values_events[0]["data"] == workflow_input
assert values_events[1]["data"]["normalized_topic"] == "langgraph stream"
assert values_events[-1]["data"]["result"] == "已处理主题：langgraph stream"
print("values 模式的协议与完整 State 演进验证通过。")


values 模式的协议与完整 State 演进验证通过。


### 结果解读

- 第一个事件是输入 State，此时两个输出字段还是空字符串。
- 第二个事件已经合并 `normalize_topic` 的返回值。
- 第三个事件包含最终 `result`。
- v2 的主要判别字段是 `type`，`ns=()` 表示事件来自根图；某些模式还可能带有额外的顶层协议字段，因此应保留完整原始对象观察。

## 3. `updates`：只观察节点本次写入的增量

`updates` 不重复发送完整 State。每个事件的 `data` 使用节点名作为键，值是该节点返回的 State 更新。并行节点位于同一个 step 时，它们的更新仍会分别发出。

In [5]:
updates_events = list(
    workflow_graph.stream(
        workflow_input,
        stream_mode="updates",
        version="v2",
    )
)

print("updates 模式的全部原始 StreamPart：")
for index, event in enumerate(updates_events, start=1):
    print(f"\n--- updates event {index} ---")
    pprint(event)


updates 模式的全部原始 StreamPart：

--- updates event 1 ---
{'data': {'normalize_topic': {'normalized_topic': 'langgraph stream'}},
 'ns': (),
 'type': 'updates'}

--- updates event 2 ---
{'data': {'build_result': {'result': '已处理主题：langgraph stream'}},
 'ns': (),
 'type': 'updates'}


In [6]:
assert len(updates_events) == 2
assert all(event["type"] == "updates" for event in updates_events)
assert updates_events[0]["data"] == {
    "normalize_topic": {"normalized_topic": "langgraph stream"}
}
assert updates_events[1]["data"] == {
    "build_result": {"result": "已处理主题：langgraph stream"}
}
print("updates 模式的节点名与增量数据验证通过。")


updates 模式的节点名与增量数据验证通过。


### `values` 与 `updates` 如何选择

- UI 每次都需要完整状态时，`values` 消费最直接。
- State 较大、只关心哪个节点改了什么时，`updates` 更节省传输量。
- 两种模式都反映图的 State 写入；它们不会自动包含节点内部尚未写入 State 的进度，这类数据应使用 `custom`。

## 4. `custom`：从节点内部主动发送业务进度

节点通过 `get_stream_writer()` 获取当前运行的 writer。写入的数据不会自动进入 State，也不要求固定 schema；调用方只有启用 `custom` 模式时才会收到这些事件。

In [7]:
custom_events = list(
    workflow_graph.stream(
        workflow_input,
        stream_mode="custom",
        version="v2",
    )
)

print("custom 模式的全部原始 StreamPart：")
for index, event in enumerate(custom_events, start=1):
    print(f"\n--- custom event {index} ---")
    pprint(event)


custom 模式的全部原始 StreamPart：

--- custom event 1 ---
{'data': {'event': 'progress', 'message': '正在规范化主题', 'node': 'normalize_topic'},
 'ns': (),
 'type': 'custom'}

--- custom event 2 ---
{'data': {'event': 'progress', 'message': '正在生成最终结果', 'node': 'build_result'},
 'ns': (),
 'type': 'custom'}


In [8]:
assert len(custom_events) == 2
assert all(event["type"] == "custom" for event in custom_events)
assert [event["data"]["node"] for event in custom_events] == [
    "normalize_topic",
    "build_result",
]
assert all(event["data"]["event"] == "progress" for event in custom_events)
print("custom 模式的两条业务进度事件验证通过。")


custom 模式的两条业务进度事件验证通过。


## 5. 一次订阅多个模式

`stream_mode` 可以接收模式列表。v2 中所有事件仍是同一种 `StreamPart` 字典，消费方直接根据 `event["type"]` 分派，不需要因为模式数量变化而切换解包方式。

In [9]:
multi_mode_events = list(
    workflow_graph.stream(
        workflow_input,
        stream_mode=["updates", "custom"],
        version="v2",
    )
)

print("updates + custom 的全部原始 StreamPart（保持到达顺序）：")
for index, event in enumerate(multi_mode_events, start=1):
    print(f"\n--- mixed event {index} ---")
    pprint(event)


updates + custom 的全部原始 StreamPart（保持到达顺序）：

--- mixed event 1 ---
{'data': {'event': 'progress', 'message': '正在规范化主题', 'node': 'normalize_topic'},
 'ns': (),
 'type': 'custom'}

--- mixed event 2 ---
{'data': {'normalize_topic': {'normalized_topic': 'langgraph stream'}},
 'ns': (),
 'type': 'updates'}

--- mixed event 3 ---
{'data': {'event': 'progress', 'message': '正在生成最终结果', 'node': 'build_result'},
 'ns': (),
 'type': 'custom'}

--- mixed event 4 ---
{'data': {'build_result': {'result': '已处理主题：langgraph stream'}},
 'ns': (),
 'type': 'updates'}


In [10]:
event_types = [event["type"] for event in multi_mode_events]
print("事件类型到达顺序（完整列表）：")
pprint(event_types)

assert event_types.count("custom") == 2
assert event_types.count("updates") == 2
assert all(set(event) >= {"type", "ns", "data"} for event in multi_mode_events)
print("多模式 v2 统一协议验证通过。")


事件类型到达顺序（完整列表）：
['custom', 'updates', 'custom', 'updates']
多模式 v2 统一协议验证通过。


## 6. `messages`：真实模型消息分块与 metadata

本节真实调用 DeepSeek Flash。即使模型节点内部使用 `invoke()`，外层图启用 `messages` 后，LangGraph 也会通过流式 callback 发出模型生成的 `AIMessageChunk`。每个事件的 `data` 是二元组：

```python
(message_chunk, metadata)
```

模型的分块边界、调用 ID 和自然语言内容每次都可能变化，因此本节先逐块输出完整原始对象，再只验证稳定协议。

In [11]:
load_dotenv(override=True)
deepseek_key_configured = bool(os.getenv("DEEPSEEK_API_KEY"))
print(
    "DEEPSEEK_API_KEY:",
    "SET（值已隐藏）" if deepseek_key_configured else "BLOCKED（未配置）",
)
if not deepseek_key_configured:
    raise RuntimeError("未配置 DEEPSEEK_API_KEY，无法运行真实 messages 流示例。")

DEEPSEEK_MODEL = "deepseek:deepseek-v4-flash"
deepseek_model = init_chat_model(
    model=DEEPSEEK_MODEL,
    temperature=0,
    extra_body={"thinking": {"type": "disabled"}},
)


def call_model(state: MessagesState) -> dict[str, list]:
    response = deepseek_model.invoke(state["messages"])
    return {"messages": [response]}


message_builder = StateGraph(MessagesState)
message_builder.add_node("call_model", call_model)
message_builder.add_edge(START, "call_model")
message_builder.add_edge("call_model", END)
message_graph = message_builder.compile()

print("真实模型客户端：", DEEPSEEK_MODEL)


DEEPSEEK_API_KEY: SET（值已隐藏）
真实模型客户端： deepseek:deepseek-v4-flash


In [12]:
message_events = []
message_input = {
    "messages": [
        HumanMessage(content="请用一句简洁的中文解释 LangGraph 的流式输出。")
    ]
}

print("messages 模式开始逐块接收；下面是每个完整原始 StreamPart：")
for index, event in enumerate(
    message_graph.stream(
        message_input,
        stream_mode="messages",
        version="v2",
    ),
    start=1,
):
    message_events.append(event)
    print(f"\n--- messages event {index} ---")
    pprint(event)


messages 模式开始逐块接收；下面是每个完整原始 StreamPart：

--- messages event 1 ---
{'data': (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--01a03f67-30c9-70d1-919b-76f782115ae2', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]),
          {'checkpoint_ns': 'call_model:26d0d566-1f1b-e383-3c7e-3a6c2415881b',
           'langgraph_checkpoint_ns': 'call_model:26d0d566-1f1b-e383-3c7e-3a6c2415881b',
           'langgraph_node': 'call_model',
           'langgraph_path': ('__pregel_pull', 'call_model'),
           'langgraph_step': 1,
           'langgraph_triggers': ('branch:to:call_model',),
           'ls_model_name': 'deepseek-v4-flash',
           'ls_model_type': 'chat',
           'ls_provider': 'deepseek',
           'ls_temperature': 0.0}),
 'ns': (),
 'type': 'messages'}

--- messages event 2 ---
{'data': (AIMessageChunk(content='Lang', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--01a03f6

In [13]:
message_chunks = [event["data"][0] for event in message_events]
message_metadata = [event["data"][1] for event in message_events]
streamed_text = "".join(str(chunk.text) for chunk in message_chunks)

print("聚合后的模型文本：")
print(streamed_text)
print("\n第一条完整 metadata：")
pprint(message_metadata[0])

assert message_events
assert all(event["type"] == "messages" for event in message_events)
assert all(isinstance(chunk, AIMessageChunk) for chunk in message_chunks)
assert streamed_text.strip()
assert all(metadata["langgraph_node"] == "call_model" for metadata in message_metadata)
assert all(metadata["ls_model_type"] == "chat" for metadata in message_metadata)
print("messages 模式的分块、聚合文本和节点 metadata 验证通过。")


聚合后的模型文本：
LangGraph 的流式输出是指：在智能体逐步执行时，实时逐块（token/节点）返回中间结果，而非等待全部完成。

第一条完整 metadata：
{'checkpoint_ns': 'call_model:26d0d566-1f1b-e383-3c7e-3a6c2415881b',
 'langgraph_checkpoint_ns': 'call_model:26d0d566-1f1b-e383-3c7e-3a6c2415881b',
 'langgraph_node': 'call_model',
 'langgraph_path': ('__pregel_pull', 'call_model'),
 'langgraph_step': 1,
 'langgraph_triggers': ('branch:to:call_model',),
 'ls_model_name': 'deepseek-v4-flash',
 'ls_model_type': 'chat',
 'ls_provider': 'deepseek',
 'ls_temperature': 0.0}
messages 模式的分块、聚合文本和节点 metadata 验证通过。


### 结果解读

- `message_chunk` 是可以逐步拼接的 `AIMessageChunk`，并不等同于最终完整 `AIMessage`。
- metadata 标明了产生分块的图节点、step、checkpoint namespace 与模型类型，可用于过滤多个并发模型节点的输出。
- 不应假设一个 chunk 就是一个词或一个 token；供应商、传输层和 LangChain 集成都可能影响分块边界。
- 如果需要最终完整消息，可以像上面的验证单元一样聚合 chunk，也可以同时订阅 `updates` 观察节点完成后写回 State 的完整消息。

## 7. `checkpoints`、`tasks` 与 `debug`：观察运行时

这三种模式关注的不是业务 UI，而是图执行与持久化机制。本节给基础图增加 `InMemorySaver`，并为每种观察模式使用独立 `thread_id`，避免多次演示共享同一条 checkpoint 时间线。

In [14]:
tracked_graph = workflow_builder.compile(checkpointer=InMemorySaver())

checkpoint_config = {"configurable": {"thread_id": "stream-checkpoints-demo"}}
tasks_config = {"configurable": {"thread_id": "stream-tasks-demo"}}
debug_config = {"configurable": {"thread_id": "stream-debug-demo"}}

print("checkpoints 配置（完整原始对象）：")
pprint(checkpoint_config)
print("tasks 配置（完整原始对象）：")
pprint(tasks_config)
print("debug 配置（完整原始对象）：")
pprint(debug_config)


checkpoints 配置（完整原始对象）：
{'configurable': {'thread_id': 'stream-checkpoints-demo'}}
tasks 配置（完整原始对象）：
{'configurable': {'thread_id': 'stream-tasks-demo'}}
debug 配置（完整原始对象）：
{'configurable': {'thread_id': 'stream-debug-demo'}}


In [15]:
checkpoint_events = list(
    tracked_graph.stream(
        workflow_input,
        checkpoint_config,
        stream_mode="checkpoints",
        version="v2",
    )
)

print("checkpoints 模式的全部原始 StreamPart：")
for index, event in enumerate(checkpoint_events, start=1):
    print(f"\n--- checkpoint event {index} ---")
    pprint(event)


checkpoints 模式的全部原始 StreamPart：

--- checkpoint event 1 ---
{'data': {'config': {'configurable': {'checkpoint_id': '1f1a17ec-317f-66f2-bfff-36c99d0d12be',
                                      'checkpoint_ns': '',
                                      'thread_id': 'stream-checkpoints-demo'}},
          'metadata': {'parents': {}, 'source': 'input', 'step': -1},
          'next': ['__start__'],
          'parent_config': None,
          'tasks': [{'id': '5811488a-fd38-ccbe-3b2b-fedb7d01d657',
                     'interrupts': (),
                     'name': '__start__',
                     'state': None}],
          'values': {}},
 'ns': (),
 'type': 'checkpoints'}

--- checkpoint event 2 ---
{'data': {'config': {'configurable': {'checkpoint_id': '1f1a17ec-3180-6b56-8000-c8a4d0a72bba',
                                      'checkpoint_ns': '',
                                      'thread_id': 'stream-checkpoints-demo'}},
          'metadata': {'parents': {}, 'source': 'loop', 'step'

In [16]:
task_events = list(
    tracked_graph.stream(
        workflow_input,
        tasks_config,
        stream_mode="tasks",
        version="v2",
    )
)

print("tasks 模式的全部原始 StreamPart：")
for index, event in enumerate(task_events, start=1):
    print(f"\n--- task event {index} ---")
    pprint(event)


tasks 模式的全部原始 StreamPart：

--- task event 1 ---
{'data': {'id': '06fbda0d-bf76-b226-62f8-32a2e025991f',
          'input': {'normalized_topic': '',
                    'result': '',
                    'topic': '  LangGraph Stream  '},
          'name': 'normalize_topic',
          'triggers': ('branch:to:normalize_topic',)},
 'ns': (),
 'type': 'tasks'}

--- task event 2 ---
{'data': {'error': None,
          'id': '06fbda0d-bf76-b226-62f8-32a2e025991f',
          'interrupts': [],
          'name': 'normalize_topic',
          'result': {'normalized_topic': 'langgraph stream'}},
 'ns': (),
 'type': 'tasks'}

--- task event 3 ---
{'data': {'id': 'a31f0029-1e11-cac2-606e-8fc0dbcb61dc',
          'input': {'normalized_topic': 'langgraph stream',
                    'result': '',
                    'topic': '  LangGraph Stream  '},
          'name': 'build_result',
          'triggers': ('branch:to:build_result',)},
 'ns': (),
 'type': 'tasks'}

--- task event 4 ---
{'data': {'error': N

In [17]:
debug_events = list(
    tracked_graph.stream(
        workflow_input,
        debug_config,
        stream_mode="debug",
        version="v2",
    )
)

print("debug 模式的全部原始 StreamPart：")
for index, event in enumerate(debug_events, start=1):
    print(f"\n--- debug event {index} ---")
    pprint(event)


debug 模式的全部原始 StreamPart：

--- debug event 1 ---
{'data': {'payload': {'config': {'configurable': {'checkpoint_id': '1f1a17ec-3294-609c-bfff-028e639d0fc9',
                                                  'checkpoint_ns': '',
                                                  'thread_id': 'stream-debug-demo'}},
                      'metadata': {'parents': {},
                                   'source': 'input',
                                   'step': -1},
                      'next': ['__start__'],
                      'parent_config': None,
                      'tasks': [{'id': 'dc476213-7918-4234-7b9f-cf4c02f28abd',
                                 'interrupts': (),
                                 'name': '__start__',
                                 'state': None}],
                      'values': {}},
          'step': -1,
          'timestamp': '2026-08-26T18:48:43.741204+00:00',
          'type': 'checkpoint'},
 'ns': (),
 'type': 'debug'}

--- debug event 2 ---
{'data':

In [18]:
checkpoint_payloads = [event["data"] for event in checkpoint_events]
task_payloads = [event["data"] for event in task_events]
debug_payload_types = [event["data"]["type"] for event in debug_events]

print("debug 内部事件类型的完整到达顺序：")
pprint(debug_payload_types)

assert checkpoint_events
assert all(event["type"] == "checkpoints" for event in checkpoint_events)
assert all(set(payload) >= {"config", "values", "next", "tasks"} for payload in checkpoint_payloads)
assert task_events
assert all(event["type"] == "tasks" for event in task_events)
assert any("input" in payload for payload in task_payloads)
assert any("result" in payload for payload in task_payloads)
assert debug_events
assert all(event["type"] == "debug" for event in debug_events)
assert {"checkpoint", "task", "task_result"} <= set(debug_payload_types)
print("checkpoints、tasks、debug 三种运行时观察协议验证通过。")


debug 内部事件类型的完整到达顺序：
['checkpoint',
 'checkpoint',
 'task',
 'task_result',
 'checkpoint',
 'task',
 'task_result',
 'checkpoint']
checkpoints、tasks、debug 三种运行时观察协议验证通过。


### 结果解读

- `checkpoints` 在输入、step 边界和图结束时发出完整 checkpoint 观察数据，包括 `values`、`next`、`tasks` 与父配置。
- `tasks` 为真正执行的节点发出开始事件和完成事件；二者共享 task ID，完成事件包含 `result`、`error` 和 interrupts。
- `debug` 的外层仍是 `type='debug'`，内部 `data['type']` 再区分 `checkpoint`、`task`、`task_result`。它信息最丰富，但体积也最大。
- `InMemorySaver` 只适合 Notebook、测试和单进程调试；它不是跨进程持久化方案。

## 8. v1 与 v2 输出形状

LangGraph 1.1.2 中 `version` 默认是 `"v1"`：

- v1 单模式直接返回该模式的 payload；
- v1 多模式返回 `(mode, payload)` 二元组；
- v2 无论单模式、多模式还是子图，都使用带判别字段的 `StreamPart` 字典。

新代码推荐显式传入 `version="v2"`，不要依赖默认值。

In [19]:
default_updates = list(
    workflow_graph.stream(workflow_input, stream_mode="updates")
)
v1_multi_events = list(
    workflow_graph.stream(
        workflow_input,
        stream_mode=["updates", "custom"],
        version="v1",
    )
)
v2_multi_events = list(
    workflow_graph.stream(
        workflow_input,
        stream_mode=["updates", "custom"],
        version="v2",
    )
)

print("默认版本 + 单模式的完整原始输出：")
pprint(default_updates)
print("\nv1 + 多模式的完整原始输出：")
pprint(v1_multi_events)
print("\nv2 + 多模式的完整原始输出：")
pprint(v2_multi_events)


默认版本 + 单模式的完整原始输出：
[{'normalize_topic': {'normalized_topic': 'langgraph stream'}},
 {'build_result': {'result': '已处理主题：langgraph stream'}}]

v1 + 多模式的完整原始输出：
[('custom',
  {'event': 'progress', 'message': '正在规范化主题', 'node': 'normalize_topic'}),
 ('updates', {'normalize_topic': {'normalized_topic': 'langgraph stream'}}),
 ('custom',
  {'event': 'progress', 'message': '正在生成最终结果', 'node': 'build_result'}),
 ('updates', {'build_result': {'result': '已处理主题：langgraph stream'}})]

v2 + 多模式的完整原始输出：
[{'data': {'event': 'progress',
           'message': '正在规范化主题',
           'node': 'normalize_topic'},
  'ns': (),
  'type': 'custom'},
 {'data': {'normalize_topic': {'normalized_topic': 'langgraph stream'}},
  'ns': (),
  'type': 'updates'},
 {'data': {'event': 'progress', 'message': '正在生成最终结果', 'node': 'build_result'},
  'ns': (),
  'type': 'custom'},
 {'data': {'build_result': {'result': '已处理主题：langgraph stream'}},
  'ns': (),
  'type': 'updates'}]


In [20]:
assert default_updates
assert all(isinstance(event, dict) and "type" not in event for event in default_updates)
assert v1_multi_events
assert all(
    isinstance(event, tuple) and event[0] in {"updates", "custom"}
    for event in v1_multi_events
)
assert v2_multi_events
assert all(
    isinstance(event, dict) and set(event) >= {"type", "ns", "data"}
    for event in v2_multi_events
)
print("默认 v1、v1 多模式与 v2 统一结构的差异验证通过。")


默认 v1、v1 多模式与 v2 统一结构的差异验证通过。


## 9. `subgraphs=True`：用 namespace 区分根图与子图

默认只返回根图事件。传入 `subgraphs=True` 后，子图内部事件也会进入同一个流。v2 使用 `ns` 表示来源：根图为 `()`，子图则包含类似 `child_graph:<运行时 ID>` 的路径段。运行时 ID 每次都会变化，因此只验证 namespace 是否为空以及稳定的节点名前缀。

In [21]:
class NestedState(TypedDict):
    text: str
    child_text: str
    final_text: str


def child_prepare(state: NestedState) -> dict[str, str]:
    return {"child_text": f"{state['text']} -> child"}


child_builder = StateGraph(NestedState)
child_builder.add_node("child_prepare", child_prepare)
child_builder.add_edge(START, "child_prepare")
child_builder.add_edge("child_prepare", END)
child_graph = child_builder.compile()


def parent_finish(state: NestedState) -> dict[str, str]:
    return {"final_text": f"{state['child_text']} -> parent"}


parent_builder = StateGraph(NestedState)
parent_builder.add_node("child_graph", child_graph)
parent_builder.add_node("parent_finish", parent_finish)
parent_builder.add_edge(START, "child_graph")
parent_builder.add_edge("child_graph", "parent_finish")
parent_builder.add_edge("parent_finish", END)
parent_graph = parent_builder.compile()


In [22]:
subgraph_events = list(
    parent_graph.stream(
        {"text": "root", "child_text": "", "final_text": ""},
        stream_mode="updates",
        subgraphs=True,
        version="v2",
    )
)

print("包含子图的全部原始 StreamPart：")
for index, event in enumerate(subgraph_events, start=1):
    print(f"\n--- subgraph event {index} ---")
    pprint(event)


包含子图的全部原始 StreamPart：

--- subgraph event 1 ---
{'data': {'child_prepare': {'child_text': 'root -> child'}},
 'ns': ('child_graph:3e135082-0507-86d6-13f9-c78715cfd424',),
 'type': 'updates'}

--- subgraph event 2 ---
{'data': {'child_graph': {'child_text': 'root -> child',
                          'final_text': '',
                          'text': 'root'}},
 'ns': (),
 'type': 'updates'}

--- subgraph event 3 ---
{'data': {'parent_finish': {'final_text': 'root -> child -> parent'}},
 'ns': (),
 'type': 'updates'}


In [23]:
root_events = [event for event in subgraph_events if event["ns"] == ()]
child_events = [event for event in subgraph_events if event["ns"] != ()]

print("根图事件（完整原始列表）：")
pprint(root_events)
print("\n子图事件（完整原始列表）：")
pprint(child_events)

assert root_events
assert child_events
assert all(event["type"] == "updates" for event in subgraph_events)
assert all(event["ns"][0].startswith("child_graph:") for event in child_events)
assert child_events[0]["data"] == {
    "child_prepare": {"child_text": "root -> child"}
}
assert root_events[-1]["data"] == {
    "parent_finish": {"final_text": "root -> child -> parent"}
}
print("根图与子图 namespace、节点增量验证通过。")


根图事件（完整原始列表）：
[{'data': {'child_graph': {'child_text': 'root -> child',
                           'final_text': '',
                           'text': 'root'}},
  'ns': (),
  'type': 'updates'},
 {'data': {'parent_finish': {'final_text': 'root -> child -> parent'}},
  'ns': (),
  'type': 'updates'}]

子图事件（完整原始列表）：
[{'data': {'child_prepare': {'child_text': 'root -> child'}},
  'ns': ('child_graph:3e135082-0507-86d6-13f9-c78715cfd424',),
  'type': 'updates'}]
根图与子图 namespace、节点增量验证通过。


## 10. `astream()`：异步消费同一套协议

`astream()` 返回异步迭代器，`stream_mode`、`version`、`subgraphs` 等参数与同步版本一致。Notebook 支持顶层 `await` / `async for`，因此可以直接运行。这里复用无模型基础图，不再产生第二次付费模型调用。

In [24]:
async_events = []
async for event in workflow_graph.astream(
    workflow_input,
    stream_mode=["updates", "custom"],
    version="v2",
):
    async_events.append(event)

print("astream() 的全部原始 StreamPart：")
for index, event in enumerate(async_events, start=1):
    print(f"\n--- async event {index} ---")
    pprint(event)


astream() 的全部原始 StreamPart：

--- async event 1 ---
{'data': {'event': 'progress', 'message': '正在规范化主题', 'node': 'normalize_topic'},
 'ns': (),
 'type': 'custom'}

--- async event 2 ---
{'data': {'normalize_topic': {'normalized_topic': 'langgraph stream'}},
 'ns': (),
 'type': 'updates'}

--- async event 3 ---
{'data': {'event': 'progress', 'message': '正在生成最终结果', 'node': 'build_result'},
 'ns': (),
 'type': 'custom'}

--- async event 4 ---
{'data': {'build_result': {'result': '已处理主题：langgraph stream'}},
 'ns': (),
 'type': 'updates'}


In [25]:
async_event_types = [event["type"] for event in async_events]
print("异步事件类型到达顺序（完整列表）：")
pprint(async_event_types)

assert async_event_types.count("custom") == 2
assert async_event_types.count("updates") == 2
assert all(set(event) >= {"type", "ns", "data"} for event in async_events)
print("astream() 与 v2 多模式协议验证通过。")


异步事件类型到达顺序（完整列表）：
['custom', 'updates', 'custom', 'updates']
astream() 与 v2 多模式协议验证通过。


## 11. `StreamEvent` 与 `astream_events()`

这里容易混淆三个名字：

- `StreamPart`：前面 `stream()` / `astream()` 在 `version="v2"` 下返回的 LangGraph 模式投影，核心字段是 `type`、`ns`、`data`。
- `StreamEvent`：LangChain Runnable callback 系统的生命周期事件，核心字段是 `event`、`name`、`run_id`、`parent_ids`、`tags`、`metadata`、`data`。
- `astream_events()`：异步遍历 `StreamEvent` 的方法。当前项目版本没有同步 `stream_events()`，也没有单数形式的 `stream_event()`。

`CompiledStateGraph` 本身实现了 Runnable 接口，因此可以调用 `astream_events()`。标准事件名遵循 `on_<runnable_type>_<start|stream|end>`，例如节点和根图产生的 `on_chain_start`、`on_chain_stream`、`on_chain_end`。

| 对比项 | `astream(..., stream_mode=...)` | `astream_events()` |
| --- | --- | --- |
| 抽象层 | LangGraph 图运行投影 | LangChain Runnable callback 生命周期 |
| 主要判别字段 | `type` | `event` 与 `name` |
| 父子关系 | `ns` 表示图/子图 namespace | `run_id` 与 `parent_ids` 表示 Runnable 调用树 |
| 适合场景 | State、模型分块、业务进度、task/checkpoint | 观察模型、工具、retriever、节点等所有内部 Runnable |
| 当前默认版本 | LangGraph 1.1.2 的 `astream()` 默认 v1 | 本项目 `astream_events()` 默认 v2 |

下面复用无模型基础图，只观察本地 Runnable 生命周期，不产生额外模型调用。动态 `run_id` 和 checkpoint namespace 只展示，不做固定值断言。

In [26]:
stream_events: list[StreamEvent] = []
async for event in workflow_graph.astream_events(
    workflow_input,
    version="v2",
):
    stream_events.append(event)

print("astream_events() 的全部原始 StreamEvent：")
for index, event in enumerate(stream_events, start=1):
    print(f"\n--- StreamEvent {index} ---")
    pprint(event)


astream_events() 的全部原始 StreamEvent：

--- StreamEvent 1 ---
{'data': {'input': {'normalized_topic': '',
                    'result': '',
                    'topic': '  LangGraph Stream  '}},
 'event': 'on_chain_start',
 'metadata': {},
 'name': 'LangGraph',
 'parent_ids': [],
 'run_id': '01a03f67-36db-7663-b787-f1d57494a059',
 'tags': []}

--- StreamEvent 2 ---
{'data': {'input': {'normalized_topic': '',
                    'result': '',
                    'topic': '  LangGraph Stream  '}},
 'event': 'on_chain_start',
 'metadata': {'langgraph_checkpoint_ns': 'normalize_topic:9edf00dd-bcc1-4907-83db-ff4dbc82126d',
              'langgraph_node': 'normalize_topic',
              'langgraph_path': ('__pregel_pull', 'normalize_topic'),
              'langgraph_step': 1,
              'langgraph_triggers': ('branch:to:normalize_topic',)},
 'name': 'normalize_topic',
 'parent_ids': ['01a03f67-36db-7663-b787-f1d57494a059'],
 'run_id': '01a03f67-36dc-72f2-a13f-2fe85bd3501c',
 'tags': ['graph

In [27]:
stream_event_names = [event["event"] for event in stream_events]
runnable_names = [event["name"] for event in stream_events]
root_run_id = stream_events[0]["run_id"]
child_events = [event for event in stream_events if event["parent_ids"]]

print("生命周期事件名的完整到达顺序：")
pprint(stream_event_names)
print("\nRunnable 名称的完整到达顺序：")
pprint(runnable_names)

assert stream_events
assert all(
    set(event) >= {
        "event",
        "name",
        "run_id",
        "parent_ids",
        "tags",
        "metadata",
        "data",
    }
    for event in stream_events
)
assert {"on_chain_start", "on_chain_stream", "on_chain_end"} <= set(
    stream_event_names
)
assert {"LangGraph", "normalize_topic", "build_result"} <= set(runnable_names)
assert stream_events[0]["parent_ids"] == []
assert child_events
assert all(root_run_id in event["parent_ids"] for event in child_events)
print("StreamEvent schema、生命周期事件和父子 Runnable 关系验证通过。")


生命周期事件名的完整到达顺序：
['on_chain_start',
 'on_chain_start',
 'on_chain_stream',
 'on_chain_end',
 'on_chain_stream',
 'on_chain_start',
 'on_chain_stream',
 'on_chain_end',
 'on_chain_stream',
 'on_chain_end']

Runnable 名称的完整到达顺序：
['LangGraph',
 'normalize_topic',
 'normalize_topic',
 'normalize_topic',
 'LangGraph',
 'build_result',
 'build_result',
 'build_result',
 'LangGraph',
 'LangGraph']
StreamEvent schema、生命周期事件和父子 Runnable 关系验证通过。


### 在生产代码中先过滤，再消费

完整事件流非常详细。`astream_events()` 支持 `include_names`、`include_types`、`include_tags` 以及对应的 `exclude_*` 参数。过滤发生在事件生产接口内部，比先接收全部事件再手工丢弃更清楚。下面只保留 `normalize_topic` 节点的事件。

In [28]:
filtered_stream_events: list[StreamEvent] = []
async for event in workflow_graph.astream_events(
    workflow_input,
    version="v2",
    include_names=["normalize_topic"],
):
    filtered_stream_events.append(event)

print("只保留 normalize_topic 后的全部原始 StreamEvent：")
for index, event in enumerate(filtered_stream_events, start=1):
    print(f"\n--- filtered StreamEvent {index} ---")
    pprint(event)


只保留 normalize_topic 后的全部原始 StreamEvent：

--- filtered StreamEvent 1 ---
{'data': {'input': {'normalized_topic': '',
                    'result': '',
                    'topic': '  LangGraph Stream  '}},
 'event': 'on_chain_start',
 'metadata': {'langgraph_checkpoint_ns': 'normalize_topic:c4e84a86-1a76-d2fe-2ee5-549bf989bf6c',
              'langgraph_node': 'normalize_topic',
              'langgraph_path': ('__pregel_pull', 'normalize_topic'),
              'langgraph_step': 1,
              'langgraph_triggers': ('branch:to:normalize_topic',)},
 'name': 'normalize_topic',
 'parent_ids': ['01a03f67-3713-73e0-abd1-bdcbf7d6853a'],
 'run_id': '01a03f67-3714-77b0-bc86-321c7044fdbb',
 'tags': ['graph:step:1']}

--- filtered StreamEvent 2 ---
{'data': {'chunk': {'normalized_topic': 'langgraph stream'}},
 'event': 'on_chain_stream',
 'metadata': {'langgraph_checkpoint_ns': 'normalize_topic:c4e84a86-1a76-d2fe-2ee5-549bf989bf6c',
              'langgraph_node': 'normalize_topic',
           

In [29]:
assert filtered_stream_events
assert all(event["name"] == "normalize_topic" for event in filtered_stream_events)
assert [event["event"] for event in filtered_stream_events] == [
    "on_chain_start",
    "on_chain_stream",
    "on_chain_end",
]
print("astream_events() 的 include_names 过滤验证通过。")


astream_events() 的 include_names 过滤验证通过。


## 12. 本教程用到的 Python async 知识

### 12.1 协程与异步迭代器不是同一种对象

- 调用 `async def` 函数得到**协程对象**；使用 `await coroutine` 等待它最终返回一个结果。
- `graph.astream()` 和 `graph.astream_events()` 返回**异步迭代器**；使用 `async for item in async_iterator` 逐个接收结果。
- 不能写 `await graph.astream(...)`：异步迭代器不是一个等待后只返回单值的协程。

### 12.2 `await` 的真实含义

`await` 在等待 I/O 时把控制权交还给当前线程的 event loop，让其他已调度任务有机会运行。它不会自动创建线程，也不等于并行计算。如果在 `async def` 中直接执行耗时的同步网络请求或 `time.sleep()`，仍会阻塞 event loop；应优先使用原生异步 API，无法替换时再考虑 `asyncio.to_thread()`。

### 12.3 `async for`、顺序与背压

`async for` 每次等待下一个事件，处理完当前循环体才请求下一个事件。这形成自然的背压：如果消费逻辑很慢，整体流也会变慢。同一次图运行中的事件保持它实际到达的先后顺序；只有显式创建多个 task 并发运行多个图时，不同流的事件才可能交错。

### 12.4 Notebook 与普通 Python 脚本

Jupyter kernel 已经运行 event loop，所以代码单元可以直接使用顶层 `await` 和 `async for`，不要再调用 `asyncio.run()`。普通 `.py` 脚本没有这个顶层便利语法，通常写成：

```python
import asyncio

async def main() -> None:
    async for event in graph.astream_events(inputs, version="v2"):
        print(event)

asyncio.run(main())
```

下面用一个小实验同时观察协程对象和 `async for` 消费过程。

In [30]:
async def collect_update_events() -> list[dict]:
    collected_events = []
    async for event in workflow_graph.astream(
        workflow_input,
        stream_mode="updates",
        version="v2",
    ):
        collected_events.append(event)
    return collected_events


collect_coroutine = collect_update_events()
print("调用 async def 后得到的完整原始对象：")
pprint(collect_coroutine)
print("inspect.iscoroutine：", inspect.iscoroutine(collect_coroutine))

collected_update_events = await collect_coroutine
print("\nawait 协程后返回的全部原始 StreamPart：")
pprint(collected_update_events)


调用 async def 后得到的完整原始对象：
<coroutine object collect_update_events at 0x1413ffca0>
inspect.iscoroutine： True

await 协程后返回的全部原始 StreamPart：
[{'data': {'normalize_topic': {'normalized_topic': 'langgraph stream'}},
  'ns': (),
  'type': 'updates'},
 {'data': {'build_result': {'result': '已处理主题：langgraph stream'}},
  'ns': (),
  'type': 'updates'}]


In [31]:
assert len(collected_update_events) == 2
assert all(event["type"] == "updates" for event in collected_update_events)
assert collected_update_events[-1]["data"] == {
    "build_result": {"result": "已处理主题：langgraph stream"}
}
print("async def → coroutine → await，以及函数内部 async for 的机制验证通过。")


async def → coroutine → await，以及函数内部 async for 的机制验证通过。


## 总结与选择建议

1. 新代码显式使用 `version="v2"`，通过 `type` 分派事件，通过 `ns` 判断根图或子图来源。
2. 页面整体状态使用 `values`，节点进度使用 `updates`，业务侧即时提示使用 `custom`。
3. 模型文本使用 `messages`，但不要依赖供应商分块边界；需要完整文本时自行聚合 `AIMessageChunk`。
4. `checkpoints`、`tasks`、`debug` 主要服务于持久化观察和排障，其中 `debug` 信息最全、开销也最大。
5. 多模式事件共享同一条时间顺序；应在一个消费循环里根据 `type` 分派，而不是分别调用图后再猜测事件先后。
6. `stream()` 与 `astream()` 的协议一致，区别只在同步迭代器和异步迭代器。
7. 需要观察 State/消息/task 投影时使用 `stream()` / `astream()`；需要观察内部 Runnable 生命周期和父子调用关系时使用 `astream_events()`。
8. Python 中协程使用 `await`，异步迭代器使用 `async for`；Notebook 可以使用顶层异步语法，普通脚本则从 `asyncio.run(main())` 启动 event loop。

本教程展示的是本地 Graph API。部署到 Agent Server 后的远程事件传输、断线重连、SSE 与前端状态管理属于另一层协议，不应直接套用本地迭代器的生命周期假设。